# TP 1 — Premiers pas avec Python
## Séance 1 : Introduction à la détection d’anomalies et de fraude

| | |
|---|---|
| **École** | ESAIP |
| **Classe** | Aix IR 5 CP |
| **Cours** | Détection d’anomalies et fraude avec Python |
| **Enseignant** | Serigne Faye |

## Contexte

Vous travaillez pour une banque. On vous fournit un historique de **5 000 transactions** par carte. Quelques-unes sont frauduleuses. Vous allez :

1. simuler et explorer ces données ;
2. mesurer l’accuracy d’un modèle naïf ;
3. tester une règle simple de détection à seuil ;
4. mesurer la **précision** et le **rappel** de cette règle ;
5. observer comment le seuil change les résultats.

**Rappels**

- Précision = TP / (TP + FP) : parmi les alertes, quelle part est une vraie fraude ?
- Rappel = TP / (TP + FN) : parmi les fraudes réelles, quelle part est détectée ?

## Étape 1 — Génération des données

Exécutez la cellule suivante sans la modifier. Elle crée un DataFrame `df` avec trois colonnes :

- `montant` : montant de la transaction en euros ;
- `heure` : heure de la journée (0 à 23) ;
- `fraude` : 1 si la transaction est frauduleuse, 0 sinon.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
n = 5000

# Transactions normales : montants log-normaux, activité surtout en journée
poids = np.array([1] * 6 + [4] * 16 + [2] * 2, dtype=float)
poids /= poids.sum()

df = pd.DataFrame({
    'montant': rng.lognormal(mean=3.5, sigma=0.6, size=n),
    'heure': rng.choice(24, size=n, p=poids),
    'fraude': 0,
})

# Injection de 25 fraudes : montants plus élevés, plutôt la nuit
idx = rng.choice(n, size=25, replace=False)
df.loc[idx, 'montant'] = rng.lognormal(mean=5.0, sigma=0.6, size=25)
df.loc[idx, 'heure'] = rng.integers(0, 6, size=25)
df.loc[idx, 'fraude'] = 1

df.head()

## Étape 2 — Exploration

Complétez les cellules ci-dessous (remplacez chaque `...`).

In [ ]:
# 1. Nombre de transactions normales (0) et frauduleuses (1)
...

# 2. Part de transactions frauduleuses (entre 0 et 1)
part_fraude = ...
print(f'Part de fraudes : {part_fraude:.2%}')

# 3. Statistiques du montant et de l'heure selon la classe (fraude = 0 ou 1)
...

# 4. Accuracy d'un modèle qui répond toujours « pas de fraude »
accuracy_naif = ...
print(f'Accuracy du modèle naïf : {accuracy_naif:.2%}')

In [ ]:
plt.figure(figsize=(8, 5))
# Nuage de points : heure en abscisse, montant en ordonnée,
# couleur selon la colonne 'fraude', échelle logarithmique sur l'axe des montants
...
plt.xlabel('Heure')
plt.ylabel('Montant (€)')
plt.show()

## Étape 3 — Une règle simple et son évaluation

On définit une règle de détection : **une alerte est levée si le montant dépasse un seuil**.

Complétez la fonction `evaluer` pour qu’elle calcule la matrice de confusion, la précision et le rappel.

In [ ]:
def evaluer(df, alerte):
    """alerte : série de 0/1 (1 = alerte levée). Affiche TP, FP, FN, TN, précision et rappel."""
    tp = ...   # alertes sur des vraies fraudes
    fp = ...   # alertes sur des transactions normales
    fn = ...   # fraudes sans alerte
    tn = ...   # transactions normales sans alerte
    precision = ...
    rappel = ...
    print(f'TP={tp}  FP={fp}  FN={fn}  TN={tn}')
    print(f'Précision={precision:.2f}  Rappel={rappel:.2f}')
    return precision, rappel

# Règle : alerte si montant > 150 €
alerte = (df['montant'] > 150).astype(int)
evaluer(df, alerte)

## Étape 4 — Influence du seuil

Testez plusieurs seuils et comparez les résultats.

In [ ]:
for seuil in [100, 150, 250, 400]:
    print(f'--- Seuil = {seuil} € ---')
    ...

## Étape 5 — Ajouter le contexte

Une transaction de 100 € est banale en pleine journée, mais plus suspecte à 3 h du matin. Testez la règle :

**alerte si le montant dépasse 100 € ET si l’heure est avant 6 h.**

In [ ]:
alerte2 = ...
evaluer(df, alerte2)

## Questions

Répondez dans les cellules de texte situées sous chaque question.

**Question 1.** Quelle est la part de transactions frauduleuses ? Que vaut l’accuracy du modèle qui répond toujours « pas de fraude » ? Que peut-on en conclure ?

*Votre réponse :*

**Question 2.** Observez le nuage de points. Où se situent les fraudes par rapport aux transactions normales ? Les distingue-t-on parfaitement ?

*Votre réponse :*

**Question 3.** Interprétez la précision et le rappel obtenus avec la règle « montant supérieur à 150 € ».

*Votre réponse :*

**Question 4.** Faites varier le seuil (100, 150, 250, 400). Comment évoluent la précision et le rappel ? Que constate-t-on ?

*Votre réponse :*

**Question 5.** Comparez la règle de l’étape 5 avec les règles à seuil de l’étape 4. De quel type d’anomalie s’agit-il (ponctuelle, contextuelle ou collective) ?

*Votre réponse :*

**Question 6.** Pour une banque, vaut-il mieux un seuil bas ou un seuil élevé ? Quels éléments faut-il prendre en compte pour décider ?

*Votre réponse :*

## Pour aller plus loin (facultatif)

1. Testez la règle « montant > 75 ET heure < 6 » puis « montant > 150 ET heure < 6 ». Que remarquez-vous ?
2. Modifiez le taux de fraude dans la génération des données (par exemple 100 fraudes au lieu de 25). Comment évoluent la précision et le rappel d’une même règle ?
3. Tracez la précision et le rappel en fonction du seuil sur un même graphique.